# Let's train a real CNN — on MNIST
MNIST = handwritten digits (0–9), 28×28 grayscale images. The "hello world" of CNNs.
Here's the full pipeline — we'll go section by section.
## Step 1 — Imports and data

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

### Convert images to tensors and normalize pixel values to [-1, 1]

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),                  # PIL image → tensor (C, H, W)
    transforms.Normalize((0.5,), (0.5,))   # mean=0.5, std=0.5 → range [-1, 1]
])

In [6]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [7]:
# Download MNIST (auto-downloads on first run, ~11MB)
train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:19<00:00, 499kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 99.5kB/s]
100%|██████████| 1.65M/1.65M [00:03<00:00, 440kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 343kB/s]


In [8]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

## Step 2 - Define the CNN

In [9]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Block 1: detect low-level features
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # → (32, 28, 28)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool  = nn.MaxPool2d(2, 2)                           # → (32, 14, 14)

        # Block 2: detect higher-level features
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # → (64, 14, 14)
        self.bn2   = nn.BatchNorm2d(64)
        # pool again                                               # → (64, 7, 7)

        # Classifier head (same as before — fully connected)
        self.fc1     = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2     = nn.Linear(128, 10)   # 10 digit classes
    
    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))  # conv → BN → relu → pool
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))  # conv → BN → relu → pool
        x = x.view(x.size(0), -1)                           # flatten
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)                                     # raw logits (no softmax)
        return x

## Step 3 — Training loop

In [10]:
model     = CNN()
criterion = nn.CrossEntropyLoss()          # handles softmax internally
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for images, labels in loader:
        optimizer.zero_grad()
        outputs = model(images)             # forward pass
        loss    = criterion(outputs, labels)
        loss.backward()                     # compute gradients
        optimizer.step()                    # update weights

        total_loss += loss.item()
        preds       = outputs.argmax(dim=1) # predicted class = highest logit
        correct    += (preds == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

## Step 4 — Evaluation + run

In [12]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [13]:
for epoch in range(5):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    test_loss,  test_acc  = evaluate(model, test_loader, criterion)
    print(f"Epoch {epoch+1}: train acc={train_acc:.3f}  test acc={test_acc:.3f}")

Epoch 1: train acc=0.946  test acc=0.986
Epoch 2: train acc=0.978  test acc=0.986
Epoch 3: train acc=0.982  test acc=0.988
Epoch 4: train acc=0.984  test acc=0.989
Epoch 5: train acc=0.987  test acc=0.990


In [14]:
model

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [15]:
torch.save(model.state_dict(), 'my_model.pth')

In [16]:
## Loading
# 1. Re-create the model architecture first
model_load = CNN()  # same class definition as before

# 2. Load the saved weights into it
model_load.load_state_dict(torch.load('my_model.pth'))

# 3. Set to eval mode (disables dropout, fixes batchnorm)
model_load.eval()

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [18]:
# A single 28x28 image tensor — needs a batch dimension added
image = test_data[0][0]           # shape: (1, 28, 28)
image = image.unsqueeze(0)        # shape: (1, 1, 28, 28) — fake batch of 1

with torch.no_grad():             # no gradients needed for inference
    output = model_load(image)         # shape: (1, 10)
    pred   = output.argmax(dim=1) # highest logit = predicted class
    print(f"Predicted digit: {pred.item()}")

Predicted digit: 7


```python
# Save after each epoch
torch.save({
    'epoch':                epoch,
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss':                 loss,
}, 'checkpoint.pth')
```



```python
# Resume training from checkpoint
checkpoint = torch.load('checkpoint.pth')

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1
```

## GPU → CPU loading
If you saved a model trained on GPU and want to load it on a CPU machine:
```python
model.load_state_dict(torch.load('my_model.pth', map_location='cpu'))
```